# AIND Ephys Preprocessing

Build an `AINDEPhysPreprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysPreprocessingTask` for each coordinate. The task itself clones [aind-ephys-preprocessing](https://github.com/AllenNeuralDynamics/aind-ephys-preprocessing) on first use, seeds its `data/` folder with the dispatch `job_*.json` files, writes a `params_obi.json` from the config, invokes `code/run_capsule.py`, and copies the resulting `preprocessed_<name>/`, `binary_<name>.json`, etc. into the single config's `coordinate_output_root` (under `obi-output/02_aind_ephys_preprocessing/grid_scan/<idx>/`).

We pass `obi-output/01_aind_ephys_dispatch/grid_scan/0/` (the first coordinate of the dispatch scan) as `dispatch_output_path`, clip to a 1-second window with `t_start`/`t_stop`, and disable motion correction (the default `dredge_fast` preset needs > 1 s of data).

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._02_preprocessing.blocks import MotionCorrection

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "scikit-learn",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 19 packages in 453ms
Uninstalled 2 packages in 13ms
Installed 2 packages in 5ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'scikit-learn'], returncode=0)

## Build the scan config

`dispatch_output_path` reads the `job_*.json` produced by notebook 01 directly from its `coordinate_output_root`.

In [3]:
dispatch_output_path = (
    Path.cwd() / "../../../obi-output/01_aind_ephys_dispatch/grid_scan/0"
).resolve()
assert dispatch_output_path.exists(), (
    f"{dispatch_output_path} not found. Run 01_aind_ephys_dispatch.ipynb first."
)

scan_config = obi.AINDEPhysPreprocessingScanConfig(
    initialize=obi.AINDEPhysPreprocessingScanConfig.Initialize(
        dispatch_output_path=dispatch_output_path,
        denoising_strategy="cmr",
        filter_type="highpass",
        min_preprocessing_duration=0.5,
        remove_out_channels=True,
        remove_bad_channels=True,
        max_bad_channel_fraction=0.5,
        t_start=0.0,
        t_stop=1.0,
        n_jobs=1,
    ),
    motion_correction=MotionCorrection(compute=False, apply=False),
)

## Generate the grid scan and run the preprocessing task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/02_aind_ephys_preprocessing/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 18:33:57,616] INFO: Seeded 1 job file(s) into /tmp/aind-ephys-preprocessing/data


[2026-04-29 18:33:57,616] INFO: Running python -u run_capsule.py --params params_obi.json --motion skip --n-jobs 1 --t-start 0.0 --t-stop 1.0


[2026-04-29 18:34:01,143] INFO: Running preprocessing with the following parameters:
	DENOISING_STRATEGY: cmr
	FILTER TYPE: highpass
	REMOVE_OUT_CHANNELS: True
	REMOVE_BAD_CHANNELS: True
	MAX BAD CHANNEL FRACTION: 0.5
	COMPUTE_MOTION: False
	APPLY_MOTION: False
	MOTION PRESET: dredge_fast
	MOTION TEMPORAL BIN S: 1
	T_START: 0.0
	T_STOP: 1.0
	MIN_DURATION FOR PREPROCESSING: 0.5
	N_JOBS: 1
Found 1 configurations


PREPROCESSING
Preprocessing recording: session0 - block0_None_recording1
	Original recording duration: 10.0 -- Clipping to 0.0-1.0 s
	Duration: 1.0 s
	Bad channel detection:
		- dead channels - 0
		- noise channels - 0
		- out channels - 0
	Removing 0 out channels
	Removing 0 channels after cmr preprocessing
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=8.01 MiB - total_memory=8.01 MiB - chunk_duration=1.00s
PREPROCESSING time: 2.98s



[PosixPath('../../../obi-output/02_aind_ephys_preprocessing/grid_scan/0')]

## Load the preprocessed recording back and confirm it's 1 s long

In [5]:
import spikeinterface.full as si

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

preprocessed_dirs = [
    p for p in coord_dir.iterdir() if p.is_dir() and p.name.startswith("preprocessed_")
]
print("Preprocessed recording dirs:", [p.name for p in preprocessed_dirs])

if preprocessed_dirs:
    rec = si.load(preprocessed_dirs[0])
    print(rec)
    print("duration_s:        ", rec.get_total_duration())
    print("num_channels:      ", rec.get_num_channels())
    print("sampling_frequency:", rec.get_sampling_frequency())

coordinate_output_root: ../../../obi-output/02_aind_ephys_preprocessing/grid_scan/0
Preprocessed recording dirs: ['preprocessed_block0_None_recording1']
BinaryFolderRecording: 70 channels - 30.0kHz - 1 segments - 30,001 samples - 1.00s - float32 dtype 
                       8.01 MiB
duration_s:         1.0000333333333333
num_channels:       70
sampling_frequency: 30000.0
